<a href="https://colab.research.google.com/github/miso-20/ESSA/blob/main/ESAA_OB_WEEK_01_1-review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **수상작 리뷰**

토스 NEXT ML CHALLENGE : 광고 클릭 예측(CTR) 모델 개발

https://dacon.io/competitions/official/236575/overview/description

## **주제**
광고 클릭 예측(CTR) 모델 개발




## **데이터**
train.parquet :

총 10,704,179개 샘플

총 119개 ('clicked' Target 컬럼 포함) 컬럼 존재
- gender : 성별
- age_group : 연령 그룹
- inventory_id : 지면 ID
- day_of_week : 주번호
- hour : 시간
- seq : 유저 서버 로그 시퀀스
- l_feat_* : 속성 정보 피처 (l_feat_14는 Ads set)
- feat_e_* : 정보영역 e 피처
- feat_d_* : 정보영역 d 피처
- feat_c_* : 정보영역 c 피처
- feat_b_* : 정보영역 b 피처
- feat_a_* : 정보영역 a 피처
- history_a_* : 과거 인기도 피처
- clicked : 클릭 여부 (Label)


test.parquet :

총 1,527,298개 샘플

총 119개 ('ID' 식별자 컬럼 포함) 컬럼 존재
- ID : 샘플 식별자
- gender : 성별
- age_group : 연령 그룹
- inventory_id : 지면 ID
- day_of_week : 주번호
- hour : 시간
- seq : 유저 서버 로그 시퀀스
- l_feat_* : 속성 정보 피처 (l_feat_14는 Ads set)
- feat_e_* : 정보영역 e 피처
- feat_d_* : 정보영역 d 피처
- feat_c_* : 정보영역 c 피처
- feat_b_* : 정보영역 b 피처
- feat_a_* : 정보영역 a 피처
- history_a_* : 과거 인기도 피처




## **코드 흐름**


### 1. 데이터 다운샘플링 & 전처리
- OOM(Out Of Memory) 등 대규모 데이터 자원 문제를 해결하기 위해 시드(Seed)를 고정하고 비율별로 클래스 언더샘플링 진행


### 2. 파생변수(Feature Engineering) 생성
- 범주형 기반: 주요 범주형 변수를 2개씩 짝지어 상호작용(Interaction) 파생변수 생성
- 시퀀스(seq) 기반: seq 데이터의 클릭 확률, 긍정/부정 빈도 수, 시퀀스 길이 등을 추출하여 수치화
- Count Encoding & 통계량: 카디널리티가 높은 변수에 대해 등장 빈도(count)를 매핑하고, inventory_id별 평균/표준편차 통계량 적용


### 3. Optuna 튜닝 및 모델 학습
- optuna의 TPESampler를 활용해 XGBoost 하이퍼파라미터(학습률, 트리 깊이 등)를 최적화하고 교차 검증을 통해 학습 진행


### 4. Soft Voting 앙상블 (결과 도출)
- 다양한 Seed와 비율로 학습된 모델 결과물들의 예측 확률을 평균 내어(Soft Voting) 최종 예측값 산출. (전체 전략에서는 XGBoost + CatBoost + FiBiNet을 결합)




**주요 코드**

In [ ]:
# Optuna를 활용한 하이퍼파라미터 탐색
def objective(trial):
    params = {
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "tree_method": "gpu_hist",
        "scale_pos_weight": scale_pos_weight,
        "eta": trial.suggest_float("eta", 0.01, 0.2, log=True),
        "max_depth": trial.suggest_int("max_depth", 4, 12),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0)
    }
    booster = xgb.train(
        params, dtrain, num_boost_round=1000,
        evals=[(dval, "valid")], early_stopping_rounds=30
    )
    # ... (AP와 LogLoss를 결합한 final_score 계산)
    return final_score

## **새롭게 알게 된 내용 / 어려운 내용 / 배울 점**

대규모 데이터 환경에서의 자원 최적화 기법
- 대규모 데이터를 다룰 때 단순히 고사양 장비에 의존하는 것이 아니라, 다운샘플링(UnderSampling) 전략과 최적화된 코드 설계를 통해 CPU만으로도 제한 시간(8시간) 내에 효율적인 추론이 가능하게 만드는 방법론을 배웠다

상호 보완적인 앙상블(Ensemble) 전략의 효과
- 단일 모델만으로는 성능 고점(0.3498)이 명확할 때, 각 평가지표에서 서로 강점이 다른 XGBoost(10:1)와 CatBoost(1:1)를 각기 다른 비율로 앙상블하고 보조로 딥러닝 모델(FiBiNet)을 더해 예측의 변동성을 최소화하는 고도화된 전략이 인상적이었다.